# 01 — Bronze Ingest: Commercial Routes

Raw CSV → **Bronze Delta** table with an *enforced* schema, ingestion metadata,
pre-write data-quality checks, and an idempotent **MERGE** upsert.

Source: `data/raw/gfl_commercial_routes.csv` (12,000 rows, 39 cols, grain = one row
per route per day). Target: `delta/bronze/commercial_routes`, partitioned by
`year`, `month`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from pyspark.sql import functions as F
from delta.tables import DeltaTable

from spark_session import build_spark, delta_path
from schema import SOURCE_SCHEMA, SOURCE_COLUMNS

spark = build_spark("01_bronze_ingest")

RAW_CSV   = os.path.abspath("../data/raw/gfl_commercial_routes.csv")
BRONZE    = delta_path("bronze", "commercial_routes", spark)
print("Spark", spark.version, "| Delta extension:", spark.conf.get("spark.sql.extensions"))
print("RAW   :", RAW_CSV)
print("BRONZE:", BRONZE)

:: loading settings :: url = jar:file:/Users/yashasvipamu/Documents/Python%20Files/temp_project/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/yashasvipamu/.ivy2.5.2/cache
The jars for the packages stored in: /Users/yashasvipamu/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d3877cc3-bacf-46bd-914b-272b67ec7d56;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.1 in central
	found io.delta#delta-storage;4.0.1 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 130ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.13;4.0.1 from central in [default]
	io.delta#delta-storage;4.0.1 from central in [default]
	org.antlr#antlr4-runtime;4.13.1 from central in [default]
	---------------------------------------------------------------------
	|                  

:: retrieving :: org.apache.spark#spark-submit-parent-d3877cc3-bacf-46bd-914b-272b67ec7d56
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/4ms)
26/06/23 20:40:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.0.3 | Delta extension: io.delta.sql.DeltaSparkSessionExtension
RAW   : /Users/yashasvipamu/Documents/Python Files/temp_project/gfl-route-profitability/data/raw/gfl_commercial_routes.csv
BRONZE: /Users/yashasvipamu/Documents/Python Files/temp_project/gfl-route-profitability/delta/bronze/commercial_routes


## Enforced schema

Types are **derived from the data**, not assumed (full evidence in `src/schema.py`
and the README data profile). Key confirmations made by direct inspection of the
raw file before fixing types:

- **`incident_type` has NO true nulls** — "no incident" is the *literal string*
  `"None"` (11,703 rows); real labels are Spill / Near-Miss / Vehicle Damage (297).
  A pandas profile is misleading here: pandas' default `na_values` coerces the
  string `"None"` → NaN, so it *looks* null; Spark and the raw bytes keep it
  verbatim. We declare it `nullable=True` (a future load could send a real null)
  but treat both `null` and `"None"` as "no incident". Bronze preserves the raw
  sentinel; **Silver** normalizes `"None"` → `null`. Every other column has zero
  nulls → `nullable=False`.
- **Numeric columns are genuinely numeric, not strings** — 6 integer columns are
  whole-number counts/flags; 23 are continuous doubles. A CSV that snuck a string
  (e.g. `"N/A"`) into a numeric column would fail the FAILFAST read below.
- **`date` parses as a real `DateType`** over 2022-01-01 … 2024-12-31.

`StructType` field order matches the CSV header exactly.

In [2]:
# Read with the enforced schema + FAILFAST: any schema drift in a FUTURE load
# (renamed col, new string in a numeric col, unparseable date) throws on read
# instead of silently coercing to null and corrupting Bronze.
raw = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .schema(SOURCE_SCHEMA)
    .csv(RAW_CSV)
)

# Ingestion metadata: when we loaded it, and from which file.
bronze_df = (
    raw
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.lit(os.path.basename(RAW_CSV)))
)

print("Schema enforced on read:")
bronze_df.printSchema()

Schema enforced on read:
root
 |-- route_date_key: string (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- region: string (nullable = true)
 |-- bu: string (nullable = true)
 |-- area: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- primary_waste_stream: string (nullable = true)
 |-- primary_customer_segment: string (nullable = true)
 |-- num_drivers: integer (nullable = true)
 |-- num_trucks: integer (nullable = true)
 |-- total_stops: integer (nullable = true)
 |-- completed_stops: integer (nullable = true)
 |-- missed_stops: integer (nullable = true)
 |-- total_distance_km: double (nullable = true)
 |-- total_fuel_used_litres: double (nullable = true)
 |-- total_labour_hours: double (nullable = true)
 |-- total_yards: double (nullable = true)
 |-- total_tonnes: double (nullable = true)
 |-- avg_reve

## Data quality checks (pre-write)

Run real checks against the parsed frame and print the **actual** numbers. Bronze
is the system-of-record copy, so we record DQ state but do not drop/repair here —
that is Silver's job. Each check prints a real count; zero issues is stated with
the real zero, not assumed.

In [3]:
row_count = bronze_df.count()

dup_keys = (
    bronze_df.groupBy("route_date_key").count()
    .filter(F.col("count") > 1).count()
)

# Null counts for every column declared non-nullable in SOURCE_SCHEMA.
required_cols = [f.name for f in SOURCE_SCHEMA.fields if not f.nullable]
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in required_cols]
null_row = bronze_df.select(*null_exprs).first().asDict()
null_offenders = {c: n for c, n in null_row.items() if n and n > 0}

completed_gt_total = bronze_df.filter(F.col("completed_stops") > F.col("total_stops")).count()
neg_revenue       = bronze_df.filter(F.col("total_revenue") < 0).count()
# Bonus integrity probes (cheap, informative):
neg_net_revenue   = bronze_df.filter(F.col("net_revenue") < 0).count()

# incident_type sentinel: "no incident" is the literal string "None" (NOT a null)
# in the raw file. Treat both null and "None" as no-incident when checking the
# flag/type consistency, otherwise the literal "None" looks like a violation.
NO_INCIDENT = (F.col("incident_type").isNull()) | (F.col("incident_type") == "None")
sentinel_none = bronze_df.filter(F.col("incident_type") == "None").count()
true_nulls    = bronze_df.filter(F.col("incident_type").isNull()).count()
flag_type_mismatch = bronze_df.filter(
    ((F.col("incident_flag") == 1) & NO_INCIDENT) |
    ((F.col("incident_flag") == 0) & ~NO_INCIDENT)
).count()

print("=== BRONZE DATA QUALITY (pre-write) ===")
print(f"row_count                          : {row_count}")
print(f"duplicate route_date_key keys      : {dup_keys}")
print(f"required cols with nulls           : {null_offenders if null_offenders else 'NONE (0 across all required columns)'}")
print(f"completed_stops > total_stops      : {completed_gt_total}")
print(f"negative total_revenue rows        : {neg_revenue}")
print(f"negative net_revenue rows (info)   : {neg_net_revenue}")
print(f"incident_type literal 'None' (info): {sentinel_none}   <- raw sentinel, normalized to null in Silver")
print(f"incident_type TRUE nulls           : {true_nulls}")
print(f"incident_flag/type mismatches      : {flag_type_mismatch}")

26/06/23 20:40:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


=== BRONZE DATA QUALITY (pre-write) ===
row_count                          : 12000
duplicate route_date_key keys      : 0
required cols with nulls           : NONE (0 across all required columns)
completed_stops > total_stops      : 0
negative total_revenue rows        : 0
negative net_revenue rows (info)   : 124
incident_type literal 'None' (info): 11703   <- raw sentinel, normalized to null in Silver
incident_type TRUE nulls           : 0
incident_flag/type mismatches      : 0


## Why Delta **MERGE** (not overwrite) for this dataset

The grain is **one row per `route_date_key`** (route × day). For commercial waste
routes, the costliest columns are **not final on the day the route runs**:

- **Disposal / tipping fees** (`disposal_cost`) arrive on the *landfill or transfer-
  station invoice*, often days to weeks after the haul. Until that invoice posts, a
  route-day's `disposal_cost`, `net_revenue`, `total_cost`, `gross_profit` and
  `gross_margin_pct` are estimates that get **restated**.
- **Billing corrections** (`total_revenue`, `avg_revenue_per_stop`) follow rebates,
  service credits, missed-pickup adjustments, and contract true-ups — the same
  `route_date_key` is re-issued with corrected dollars.
- **Late operational edits** — a missed-stop reclassified as completed, an incident
  logged after shift — also restate an existing key.

This is textbook **late-arriving / restated data on a stable natural key**. The
correct semantics are *upsert*: when a `route_date_key` reappears with new values,
**update that row in place**; when it's new, insert it.

- `overwrite` would force us to rewrite the whole table (or whole partitions) and
  re-load already-correct history just to patch a handful of restated keys —
  expensive and fragile.
- `append` would create **duplicate keys** the moment a disposal invoice restates a
  day, breaking the grain.
- **`MERGE ... WHEN MATCHED UPDATE WHEN NOT MATCHED INSERT`** keyed on
  `route_date_key` expresses exactly the business reality, is **idempotent**
  (re-running the same load is a no-op), and touches only the affected files. Delta's
  ACID guarantees mean a partial failure mid-correction never leaves Bronze in a
  torn state.

We additionally `whenMatchedUpdate` only the mutable measure/cost columns + refresh
`ingestion_ts`, leaving the natural key and date dimensions stable.

In [4]:
# Create the Bronze Delta table if absent (partitioned), so MERGE is the SOLE
# data path on every run -> the transaction history shows CREATE then MERGE,
# and re-running the notebook is a true idempotent upsert (no duplicate keys).
(
    DeltaTable.createIfNotExists(spark)
    .location(BRONZE)
    .addColumns(bronze_df.schema)
    .partitionedBy("year", "month")
    .execute()
)

bronze_tbl = DeltaTable.forPath(spark, BRONZE)

# Mutable columns to refresh on a restatement (everything except the natural key
# and the immutable date dimensions). ingestion_ts is bumped so we know when a
# key was last restated.
immutable = {"route_date_key", "date", "year", "month", "quarter", "day_of_week",
             "region", "bu", "area", "route_id"}
update_set = {c: F.col(f"s.{c}") for c in bronze_df.columns if c not in immutable}

(
    bronze_tbl.alias("t")
    .merge(bronze_df.alias("s"), "t.route_date_key = s.route_date_key")
    .whenMatchedUpdate(set=update_set)
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE complete.")

MERGE complete.


26/06/23 20:41:01 WARN MapPartitionsRDD: RDD 76 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


## Read back + transaction log

Read the Bronze table back from disk and dump the real Delta `DESCRIBE HISTORY`
(the transaction log) — this is the audit trail that makes restatements traceable.

In [5]:
bronze_read = spark.read.format("delta").load(BRONZE)
print("Bronze row count (read back):", bronze_read.count())
print("Distinct route_date_key     :", bronze_read.select("route_date_key").distinct().count())
print("Partitions on disk (year/month sample):")
bronze_read.select("year", "month").distinct().orderBy("year", "month").show(6)

print("=== DESCRIBE HISTORY delta/bronze/commercial_routes ===")
(
    spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE}`")
    .select("version", "timestamp", "operation", "operationMetrics")
    .show(truncate=False)
)

Bronze row count (read back): 12000


Distinct route_date_key     : 12000
Partitions on disk (year/month sample):


+----+-----+
|year|month|
+----+-----+
|2022|    1|
|2022|    2|
|2022|    3|
|2022|    4|
|2022|    5|
|2022|    6|
+----+-----+
only showing top 6 rows
=== DESCRIBE HISTORY delta/bronze/commercial_routes ===


+-------+-----------------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation   |operationMetrics                                                                                                                                                                                                                     

In [6]:
spark.stop()
print("Spark stopped.")

Spark stopped.
